# Wk10 · Day 1 · Chunking & Corpus Structuring

**Date:** Tue Apr 28, 2026  
**Anchor question:** *Why does bad chunking damage retrieval more than swapping the LLM?*  
**DocuMentor demo step:** v0 → v1 (content-type-aware splitter + metadata + tiktoken)  
**Your step on Study Assistant v2.0:** repair Wk9 chunks → add content_type metadata → token-aware sizing → log BEFORE/AFTER retrieval

---

## Scenario reminder

You are continuing the **PariShiksha NCERT Study Assistant** from Week 9. The basic version retrieves chapters but loses worked-example structure, splits diagrams from captions, and has no metadata to filter on. Today we fix the chunking — the Wk10 spine starts here.

By end of session you have:
- A re-chunked NCERT corpus with `content_type` metadata
- Token-aware chunk sizing using `tiktoken`
- A BEFORE/AFTER retrieval log on 5 of your Wk9 questions
- One concrete failure case demonstrated and (in Stretch) fixed


## 0. Setup

Run this cell once. If anything fails, talk to TA before moving on.


In [ ]:
# Pinned versions — match cohort environment
# !pip install -q "langchain==0.3.*" "langchain-community==0.3.*" "langchain-openai==0.2.*" \
#                  "chromadb==0.5.*" tiktoken rank_bm25 python-dotenv

import os, json, re
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv
load_dotenv()  # expects .env with OPENAI_API_KEY etc.

# Wk9 corpus location — adjust if your repo layout differs
CORPUS_DIR = Path("./corpus")           # NCERT chapters from Wk9
WK9_CHUNKS = Path("./wk9_chunks.json")  # your Wk9 chunk dump (optional, for diff)

assert CORPUS_DIR.exists(), f"Expected {CORPUS_DIR} from Wk9. Recreate it before continuing."
print(f"Corpus files: {sorted(p.name for p in CORPUS_DIR.iterdir())}")

## 1. Inspect what you actually have

Before chunking decisions, look at the corpus. NCERT chapters are not uniform prose — they have section headers, worked examples, in-text questions, definitions, and tables. A chunking strategy that ignores this structure is the bug we are fixing.


In [ ]:
# Load one chapter raw and look at it
chapter_path = next(CORPUS_DIR.glob("*.md"))   # or .txt — adapt to your Wk9 format
text = chapter_path.read_text(encoding="utf-8")
print(f"Chapter: {chapter_path.name}")
print(f"Total chars: {len(text):,}")
print(f"First 600 chars:\n{text[:600]}\n...")

In [ ]:
# Quick structure scan — find headings, examples, questions
headings = re.findall(r"^#+\s+.*", text, flags=re.MULTILINE)
worked_examples = re.findall(r"(?i)example\s+\d+", text)
in_text_q = re.findall(r"(?i)(?:exercise|in[- ]text question)[^\n]{0,50}", text)

print(f"Headings found: {len(headings)}")
for h in headings[:8]: print(" ", h)
print(f"\n'Example N' references: {len(worked_examples)}")
print(f"In-text questions: {len(in_text_q)}")

## 2. The Wk9 baseline — naive recursive splitter

This is roughly what most Wk9 submissions did. Run it. We will diagnose why it loses structure.


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)

naive_chunks = []
for path in CORPUS_DIR.glob("*.md"):
    text = path.read_text(encoding="utf-8")
    for i, c in enumerate(naive_splitter.split_text(text)):
        naive_chunks.append({
            "id": f"{path.stem}_naive_{i:04d}",
            "source": path.name,
            "text": c,
        })

print(f"Naive chunks total: {len(naive_chunks)}")
print(f"\nSample chunk:\n{naive_chunks[5]['text'][:400]}...")

### Diagnose the bug

Look at the sample above. Common Wk9 failures:
- A worked example **starts** in one chunk and **ends** in the next
- A question **header** is in one chunk, the **answer** in the next
- Tables get **split mid-row**
- No way to know whether a chunk is a *concept paragraph*, a *worked example*, or a *question*

This is what we fix in §3.


## 3. Content-type-aware chunking + metadata

Three branches: **prose** / **worked_example** / **question_or_exercise**. Each gets the splitter that suits it.


In [ ]:
# Detect content type per region. Heuristics — adjust for your chapter style.
EXAMPLE_RE   = re.compile(r"(?im)^\s*(?:###?\s*)?example\s+\d+", )
QUESTION_RE  = re.compile(r"(?im)^\s*(?:###?\s*)?(?:exercise|in[- ]text question|q\.?\s*\d+)")
HEADING_RE   = re.compile(r"(?m)^#+\s+.*$")

def label_region(snippet: str) -> str:
    if EXAMPLE_RE.search(snippet):
        return "worked_example"
    if QUESTION_RE.search(snippet):
        return "question_or_exercise"
    return "prose"

def section_title(text_so_far: str) -> str:
    """Last heading seen so far — used as section metadata."""
    headings = HEADING_RE.findall(text_so_far)
    return headings[-1].strip("# ").strip() if headings else "unknown"


In [ ]:
# Split by blank-line blocks first, then re-chunk each block by content type
prose_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
example_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=100)  # keep examples whole-ish
question_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

structured_chunks = []
for path in CORPUS_DIR.glob("*.md"):
    text = path.read_text(encoding="utf-8")
    blocks = re.split(r"\n\s*\n", text)
    cumulative = ""
    for blk in blocks:
        cumulative += blk + "\n\n"
        ctype = label_region(blk)
        section = section_title(cumulative)
        splitter = {"prose": prose_splitter, "worked_example": example_splitter,
                    "question_or_exercise": question_splitter}[ctype]
        for i, c in enumerate(splitter.split_text(blk)):
            structured_chunks.append({
                "id": f"{path.stem}_{ctype}_{len(structured_chunks):04d}",
                "source": path.name,
                "section": section,
                "content_type": ctype,
                "text": c,
            })

print(f"Structured chunks total: {len(structured_chunks)}")
print(f"By type: {Counter(c['content_type'] for c in structured_chunks)}")
print(f"\nSample worked_example chunk:")
ex = next(c for c in structured_chunks if c['content_type'] == 'worked_example')
print(f"section: {ex['section']}\n---\n{ex['text'][:500]}...")

## 4. Token-aware sizing

`chunk_size=800` characters means **different things** for different content. A code-heavy block has more characters per token than dense prose. Always size in tokens for embedding budget predictability.


In [ ]:
import tiktoken
enc = tiktoken.encoding_for_model("text-embedding-3-small")

# Audit your chunks — distribution of tokens per chunk
token_counts = [len(enc.encode(c["text"])) for c in structured_chunks]
print(f"Token count: min={min(token_counts)} | median={sorted(token_counts)[len(token_counts)//2]} | max={max(token_counts)}")

# Flag outliers — chunks that may have lost a sentence at the boundary
oversized = [c for c, t in zip(structured_chunks, token_counts) if t > 400]
print(f"Oversized chunks (>400 tokens): {len(oversized)}")

## 5. Demonstrate the failure mode

Pick one of your Wk9 questions where retrieval previously misfired. Compare what the naive splitter returned vs what the structured splitter returns. Don't fix it yet — see it.


In [ ]:
from rank_bm25 import BM25Okapi

def build_bm25(chunks):
    tokens = [re.findall(r"\w+", c["text"].lower()) for c in chunks]
    return BM25Okapi(tokens), tokens

def retrieve_bm25(query, bm25, chunks, k=3):
    q_tokens = re.findall(r"\w+", query.lower())
    scores = bm25.get_scores(q_tokens)
    ranked = sorted(zip(scores, chunks), key=lambda x: -x[0])[:k]
    return ranked

bm25_naive, _ = build_bm25(naive_chunks)
bm25_struct, _ = build_bm25(structured_chunks)

# Use one of your Wk9 evaluation questions
QUERY = "What is the difference between distance and displacement?"  # replace with yours

print("=== NAIVE chunks (Wk9 baseline) ===")
for score, c in retrieve_bm25(QUERY, bm25_naive, naive_chunks, k=3):
    print(f"[{score:.2f}] {c['id']}  →  {c['text'][:160]}...")

print("\n=== STRUCTURED chunks (Wk10) ===")
for score, c in retrieve_bm25(QUERY, bm25_struct, structured_chunks, k=3):
    ct = c['content_type']
    print(f"[{score:.2f}] [{ct}] {c['id']}  →  {c['text'][:160]}...")

## 6. Persist your chunks for tomorrow

Wednesday you embed these. Save once, embed once. Don't re-chunk every save.


In [ ]:
out_path = Path("wk10_chunks.json")
out_path.write_text(json.dumps(structured_chunks, indent=2), encoding="utf-8")
print(f"Saved {len(structured_chunks)} chunks → {out_path}")

---

## 🎯 Student tasks (Tue M/W/F drop)

### CORE — Foundation reset
1. Re-run §3 on **all** your Wk9 chapters. Confirm `content_type` distribution is reasonable (no chapter is 100% prose).
2. Pick **5 questions** from your Wk9 eval set. For each, log BEFORE (naive_chunks) vs AFTER (structured_chunks) top-3 retrieval.
3. Write a **2-paragraph** note: did `content_type` filtering help on any question? Where did it hurt?
4. Commit `wk10_chunks.json` + `chunking_diff.md` to your repo. Push.

### STRETCH — Production edition
1. Implement a **second variant**: `semantic_splitter` using sentence embeddings + cosine drop detection (`langchain_experimental.text_splitter.SemanticChunker`).
2. Hand-build a **10-question micro-eval set** (mix of direct, paraphrased, and worked-example-recall questions).
3. Run BM25 retrieval with **all three splitters** (naive / content-type / semantic). Score top-1 hit rate per question.
4. Write a **1-page memo**: which won, why, and where each fails. Commit `chunking_compare.md`.

### Hot Extension (optional, for those done with Stretch)
- Detect tables in NCERT pages with `pdfplumber` (if working from PDFs) or markdown table regex. Index tables as separate chunks with structured metadata. Show retrieval works on a table-specific question.


## 🏭 Where this fails in industry

What we did today is the **right starting point**. What we did not do, that real teams do:

- **Layout-aware chunking** with Unstructured.io or LayoutLM — uses visual layout (font size, position, table boundaries) to detect structure on PDFs. Critical for financial filings and research papers.
- **Domain-specific splitters** — legal teams chunk by clause, medical teams by section heading + drug name, finance teams by table-row.
- **Versioned chunking strategies** — when you change splitter logic, your old embeddings are now misaligned. Production teams version chunks the way code teams version migrations.
- **Async re-chunking pipelines** — when source docs update, you need incremental re-chunk + re-embed without taking the system down.

These are out of scope this week. Mention in your reflection if you're interested.


## 🔥 Interview drill (close of session)

1. **(Easy)** Difference between chunk size and chunk overlap?
2. **(Medium, hot)** Your retriever returns a chunk that starts mid-sentence and ends mid-sentence. The LLM still produces a fluent answer. Why is this dangerous? How would you catch it in eval?
3. **(Hard, case-study)** Your Wk9 system had recall@5 = 0.78. After today's structured chunking, recall@5 = 0.71. Some questions got worse. Diagnose three plausible causes — do **not** say "the new splitter is bad". Explain how you would isolate each cause.

Write your best 30-second answer to **#2** in your `reflection.md` before you leave today.
